## Setup

In [1]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import warnings
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

warnings.filterwarnings("ignore")
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(hf_token)

!pip install wandb -q
import wandb
wandb_token = secrets.get_secret("WANDB")
wandb.login(key=wandb_token)

import pyarrow as pa
import pyarrow.parquet as pq


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alex_dimmock (alex_dimmock-nas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Model, processor, and loss-config fix

In [2]:
!pip install transformers datasets evaluate jiwer -q

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer, TrainerCallback
from datasets import load_dataset, Dataset, Audio, get_dataset_config_names, get_dataset_split_names
from dataclasses import dataclass
from evaluate import load as load_metric
import os, torch
import unicodedata
import re
import numpy as np
import subprocess
import time

# Load pretrained characters from basque fine-tune for token characters
processor = Wav2Vec2Processor.from_pretrained("stefan-it/wav2vec2-large-xlsr-53-basque")

print("\nBasque token vocabulary:")
print(processor.tokenizer.get_vocab())

# Load base facebook model to fine-tune
ssl_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-xlsr-53", vocab_size=len(processor.tokenizer))

ssl_model.freeze_feature_encoder()

# Fix loss reduction and pad token — must happen after ssl_model is created.
# ctc_loss_reduction="sum" (the HF default) caused impossible negative loss on long batches;
# "mean" is the numerically stable standard for CTC.
ssl_model.config.ctc_loss_reduction = "mean"
ssl_model.config.pad_token_id = processor.tokenizer.pad_token_id

# Text cleaner (used both for training labels and WER normalisation)
def clean_text(t):
    t = t.lower()
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"[^\w\sñíáéóúü]", "", t)  # keep letters only
    t = re.sub(r"\s+", " ", t).strip()
    return t


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.2 MB/s eta 0:00:00:00:01


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Preprocessing

In [3]:
def preprocess(sample):
    '''
    Preprocess: takes a sample, separates the audio, sampling rate, input and labels
    Args: sample
    Returns: dict with input values and labels
    '''
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    inputs = processor(audio, sampling_rate=sr)
    
    text = clean_text(sample["sentence"])
    labels = processor(text=[text]).input_ids[0]
    
    return {
        "input_values": inputs.input_values[0],
        "labels": labels
    }

## Training callbacks

In [4]:
# Debug and Progress Callbacks — used by the real training run so long
# sessions are diagnosable from the saved Kaggle log, not just the live view.

class HeartbeatCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 100 == 0:
            print(f"[HEARTBEAT] step {state.global_step}, {time.strftime('%H:%M:%S')}", flush=True)

class DebugCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"[DEBUG {time.strftime('%H:%M:%S')}] Epoch {state.epoch} training done, starting eval...", flush=True)

    def on_evaluate(self, args, state, control, **kwargs):
        print(f"[DEBUG {time.strftime('%H:%M:%S')}] Eval complete for epoch {state.epoch}", flush=True)

    def on_save(self, args, state, control, **kwargs):
        print(f"[DEBUG {time.strftime('%H:%M:%S')}] Checkpoint saved at step {state.global_step}", flush=True)

class GPUCheckCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 200 == 0:
            result = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu", "--format=csv,noheader"], capture_output=True, text=True)
            print(f"[GPU] step {state.global_step}: {result.stdout.strip()}", flush=True)

class PrintLossCallback(TrainerCallback):
    # The Trainer's live loss table is a Jupyter widget and does NOT get
    # captured in Kaggle's saved log output — only explicit print() survives.
    # This callback makes sure loss/eval numbers are always in the saved log.
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            print(logs, flush=True)


## Data collator

In [5]:
@dataclass
class CTCDataCollator:
    processor: Wav2Vec2Processor

    def __call__(self, features):
        # Pad input_values
        input_features = [
            {"input_values": f["input_values"]}
            for f in features]
            
        batch = self.processor.pad(input_features, padding=True, return_tensors="pt")

        # Pad labels with PAD token
        label_features = [f["labels"] for f in features]
        labels_batch = self.processor.tokenizer.pad(
            {"input_ids": label_features}, padding=True, return_tensors="pt"
        )
        # Replace PAD token id with -100 so loss ignores it
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id, -100
        )
        batch["labels"] = labels
        return batch

data_collator = CTCDataCollator(processor=processor)

## WER metric

In [6]:
wer_metric = load_metric("wer")
cer_metric = load_metric("cer")

def compute_metrics(pred):
    logits = pred.predictions
    pred_ids = np.argmax(logits, axis=-1)
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    print(f"[EVAL] Decoding {len(pred_ids)} predictions...", flush=True)
    pred_str = processor.batch_decode(pred_ids, group_tokens=True)
    label_str = processor.batch_decode(label_ids, group_tokens=False)
    
    # Normalize before WER — strips case/punctuation differences that aren't real errors
    pred_str = [clean_text(s) for s in pred_str]
    label_str = [clean_text(s) for s in label_str]
    
    print(f"[EVAL] Decoding complete. Showing sample predictions:", flush=True)
    for i in range(min(8, len(pred_str))):
        print(f"  [{i}] PRED:  '{pred_str[i]}'", flush=True)
        print(f"      TRUE:  '{label_str[i]}'", flush=True)
        
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    
    print(f"[EVAL] WER computed: {wer}", flush=True)
    print(f"[EVAL] CER computed: {cer}", flush=True)
    return {"wer": wer, "cer": cer}

## Load training data (swap hour size here)

In [ ]:
'''# Load training dataset — SWAP the .take(N) and filename between hour sizes:
#   10h  -> .take(6000)
#   50h  -> .take(30500)
#   100h -> .take(61600)  (use the chunked parquet-writer version for 100h — see below)
#   200h -> .take(128000) (use the chunked parquet-writer version for 200h — see below)

raw = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True).take(61600)

def gen():
    for s in raw:
        ex = preprocess(s)
        if len(ex["input_values"]) < 320000:
            yield ex

cv_train = Dataset.from_generator(gen)
print(f"Dataset size: {len(cv_train)}")'''

In [ ]:
# --- 100h/200h variant (chunked write to parquet to avoid holding it all in memory) ---
# Only needed for the 100h/200h run — for 10h/50h, the cell above is enough.

print("Starting dataset stream...")
ds_200 = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True).take(128000)
writer = None
chunk_input = []
chunk_labels = []
chunk_size = 100
for i, s in enumerate(ds_200):
    p = preprocess(s)
    if len(p["input_values"]) < 320000 and len(p["input_values"]) // 320 > len(p["labels"]):
        chunk_input.append(p["input_values"].astype(np.float16).tolist())
        chunk_labels.append(p["labels"])
    if len(chunk_input) == chunk_size:
        table = pa.table({"input_values": chunk_input, "labels": chunk_labels})
        if writer is None:
            writer = pq.ParquetWriter("/kaggle/working/train_200h.parquet", table.schema)
        writer.write_table(table)
        chunk_input, chunk_labels = [], []
        print(f"Written {i} samples", flush=True)
if chunk_input:
    table = pa.table({"input_values": chunk_input, "labels": chunk_labels})
    if writer is None:
        writer = pq.ParquetWriter("/kaggle/working/train_200h.parquet", table.schema)
    writer.write_table(table)
writer.close()

# Disk cleanup
import gc, shutil
del ds_200, chunk_input, chunk_labels, table
gc.collect()
if os.path.exists("/kaggle/working/wav2vec2-basque-200h"):
    shutil.rmtree("/kaggle/working/wav2vec2-basque-200h")
    print("Cleared old output dir")

cv_train = Dataset.from_parquet("/kaggle/working/train_200h.parquet")
print(f"Dataset size: {len(cv_train)}")
os.remove("/kaggle/working/train_200h.parquet")
print("Deleted parquet file to free disk space")


## Load validation data

In [ ]:
# Validation set
cv_dev_raw = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="dev_cv", streaming=True).take(500)
cv_dev = []
for i, sample in enumerate(cv_dev_raw.map(preprocess)):
    cv_dev.append(sample)
    if i % 50 == 0:
        print(f"Loaded {i} val samples")
print(f"Done: {len(cv_dev)} val samples")


## Resume from a checkpoint (optional)

In [ ]:
### REQUIRED ONLY IF RESUMING A REAL CHECKPOINT ACROSS KAGGLE SESSIONS ###
# Skip this cell entirely for a fresh first run.
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="alexdimmock/wav2vec2-basque-200h",   # <- change to match the hour size you're running
    allow_patterns=["last-checkpoint/*"],
    local_dir="/kaggle/working/checkpoint",
    token=hf_token,
    #ignore_patterns=["optimizer.pt"]  # dropped to save disk — optimizer resets fresh on each resume
)

## Training arguments

In [ ]:
# Training arguments — update output_dir / run_name / hub_model_id per hour-size run
training_args = TrainingArguments(
    output_dir="/kaggle/working/wav2vec2-basque-200h",
    report_to="wandb",
    run_name="basque-200h-fixed-run1",
    per_device_train_batch_size=2,
    eval_strategy="epoch",
    bf16=True,
    learning_rate=5e-4,
    warmup_steps=200,
    logging_strategy="steps",
    logging_steps=500,
    save_strategy="steps",
    save_steps=10000,
    num_train_epochs=5,
    save_total_limit=2,
    push_to_hub=True,
    hub_model_id="alexdimmock/wav2vec2-basque-200h",
    hub_strategy="checkpoint",
    hub_token=hf_token,
)


## Eval Setup (Optional)

In [ ]:
'''### REQUIRED ONLY IF RUNNING EVAL ON A PREV FINE-TUNED MODEL ###
# Overwriting the freshly-initialized ssl_model from Setup
ssl_model = Wav2Vec2ForCTC.from_pretrained("/kaggle/working/checkpoint/last-checkpoint")'''

## Train

In [ ]:
# Add/drop train_dataset=cv_train if just for eval
trainer = Trainer(
    model=ssl_model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=cv_train,
    eval_dataset=cv_dev,
    processing_class=processor.feature_extractor,
    callbacks=[HeartbeatCallback(), DebugCallback(), GPUCheckCallback(), PrintLossCallback()]
)

# Sanity check before committing to a long run — confirms Trainer is seeing
# the full dataset (should be ~len(cv_train)/batch_size steps per epoch)
# Comment out during eval
print("train dataloader length:", len(trainer.get_train_dataloader()))

results = trainer.train()  # add resume_from_checkpoint="/kaggle/working/checkpoint/last-checkpoint" if resuming training
print(results)

# ============================================================
# DEBUG / SCRATCH — not part of the main pipeline.
# Nothing below this point is required for a normal run.
# ============================================================

### Determine sample count for a target number of hours

In [9]:
'''### TO DETERMINE NUMBER OF SAMPLES REQUIRED FOR NUMBER OF HOURS

ds = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True)

total_duration = 0
count = 0
for sample in ds:
    total_duration += len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"]
    count += 1
    if count % 500 == 0:
        print(f"{count} samples, {total_duration/3600:.2f} hours so far")
    if total_duration >= 200 * 3600:  # stop at 10 hours
        break

print(f"200 hours reached at sample {count}")'''

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

500 samples, 0.84 hours so far
1000 samples, 1.67 hours so far
1500 samples, 2.49 hours so far
2000 samples, 3.33 hours so far
2500 samples, 4.16 hours so far
3000 samples, 4.98 hours so far
3500 samples, 5.80 hours so far
4000 samples, 6.63 hours so far
4500 samples, 7.45 hours so far
5000 samples, 8.29 hours so far
5500 samples, 9.14 hours so far
6000 samples, 9.98 hours so far
6500 samples, 10.80 hours so far
7000 samples, 11.63 hours so far
7500 samples, 12.43 hours so far
8000 samples, 13.27 hours so far
8500 samples, 14.08 hours so far
9000 samples, 14.88 hours so far
9500 samples, 15.71 hours so far
10000 samples, 16.51 hours so far
10500 samples, 17.34 hours so far
11000 samples, 18.16 hours so far
11500 samples, 18.95 hours so far
12000 samples, 19.76 hours so far
12500 samples, 20.58 hours so far
13000 samples, 21.39 hours so far
13500 samples, 22.18 hours so far
14000 samples, 23.00 hours so far
14500 samples, 23.84 hours so far
15000 samples, 24.65 hours so far
15500 sample

### Check disk space

In [ ]:
''' ### TO DETERMINE DISK SPACE USED
!df -h /kaggle/working          # overall disk space available
!du -sh ~/.cache/huggingface/*  # what's cached from datasets/models
!df -h'''

### Smoke test (cheap sanity check before a full run)

In [ ]:
'''# Smoke-test args — quick, cheap sanity check before committing to a full run.
# Bump max_steps up (e.g. 300) once the fast 20-step version looks healthy.
smoke_args = TrainingArguments(
    output_dir="/kaggle/working/smoke-test",
    per_device_train_batch_size=2,
    eval_strategy="steps",
    eval_steps=100,
    bf16=True,
    learning_rate=5e-4,
    logging_strategy="steps",
    logging_steps=50,
    max_steps=300,
    report_to="none",
    push_to_hub=False,
)
'''

In [ ]:
'''### SMOKE TEST — run before a full hour-size training run
trainer = Trainer(
    model=ssl_model,
    args=smoke_args,
    data_collator=data_collator,
    train_dataset=cv_train,      # or cv_train_10h etc, whatever's loaded
    eval_dataset=cv_dev,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[PrintLossCallback()],  # ensures loss table survives Kaggle's saved log
)

trainer.train()

# Full loss/eval history, in case anything didn't print live
for entry in trainer.state.log_history:
    print(entry)
'''

### Overfit-on-small-subset debug

In [ ]:
'''### DEBUG ON SMALL OVERFITTED DATASET

from datasets import load_dataset, Dataset

raw = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True).take(500)
train_list = [preprocess(s) for s in raw]
cv_train_10h = Dataset.from_list(train_list)

# Quick sanity check: print what labels actually look like now
print("labels sample:", cv_train_10h[0]["labels"])
print("decoded:", processor.tokenizer.decode(cv_train_10h[0]["labels"]))

training_args = TrainingArguments(
    output_dir="./debug-overfit",
    per_device_train_batch_size=2,
    learning_rate=4e-4,   # was 1e-4
    max_steps=5000,       # was 200
    logging_steps=50,
    eval_strategy="no",
    save_strategy="steps",
    bf16=True,
)

trainer = Trainer(
    model=ssl_model,
    args=training_args,
    train_dataset=cv_train_10h,
    data_collator=data_collator,
    processing_class=processor,
)

trainer.train()

# Check a prediction after training
import torch, numpy as np
sample = cv_train_10h[10]
input_tensor = torch.tensor([sample["input_values"]]).to(ssl_model.device)
with torch.no_grad():
    logits = ssl_model(input_tensor).logits
pred_ids = logits.argmax(-1)
print("PRED:", processor.batch_decode(pred_ids)[0])
print("TRUE:", processor.tokenizer.decode(sample["labels"]))'''



'''try wer function on debug code'''

### Resume-from-checkpoint debug test

In [ ]:
'''# Debug Training arguments
training_args = TrainingArguments(
    output_dir="/kaggle/working/wav2vec2-basque-100h",
    report_to="wandb",
    run_name="debug-resume-test",
    per_device_train_batch_size=2,
    eval_strategy="epoch",#####CHANGE THIS TO epoch POST GPU ACCESS
    bf16=True,#####CHANGE THIS TO True POST GPU ACCESS
    learning_rate=5e-4,
    warmup_steps=200,
    logging_strategy="steps",
    logging_steps=500,#####CHANGE THIS TO 500 POST GPU ACCESS
    save_strategy="epoch",#####CHANGE THIS TO epoch POST GPU ACCESS
    num_train_epochs=5,
    save_total_limit=2,
    push_to_hub=False,#####CHANGE THIS TO True POST GPU ACCESS
    hub_model_id="alexdimmock/wav2vec2-basque-100h",
    hub_strategy="checkpoint",
    hub_token=hf_token,
    max_steps=1,
    use_cpu=True
    )

from datasets import Dataset
dummy_ds = Dataset.from_dict({"input_values": [[0.0]*16000]*4, "labels": [[1,2,3]]*4})

dummy_eval_ds = Dataset.from_dict({"input_values": [[0.0]*16000]*2, "labels": [[1,2,3]]*2})

trainer = Trainer(
    model=ssl_model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dummy_ds,
    eval_dataset=dummy_eval_ds,
    processing_class=processor.feature_extractor,
)

trainer.train(resume_from_checkpoint="/kaggle/working/checkpoint/last-checkpoint")
print("global_step after resume attempt:", trainer.state.global_step)'''

### WER function debug (evaluate an existing checkpoint on a few samples)

In [ ]:
'''### DEBUG WER FUNCTION
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/kaggle/working/debug-wer",
    per_device_eval_batch_size=4,
    report_to="none",   # skip wandb for this quick test
)

# Load your existing 10h checkpoint instead of retraining
model = Wav2Vec2ForCTC.from_pretrained("alexdimmock/wav2vec2-basque-10h")
processor = Wav2Vec2Processor.from_pretrained("stefan-it/wav2vec2-large-xlsr-53-basque")

cv_dev_raw = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="dev_cv", streaming=True).take(20)
cv_dev = [preprocess(s) for s in cv_dev_raw]

# Small eval slice — e.g. first 20 samples
small_eval = cv_dev.select(range(20)) if hasattr(cv_dev, "select") else cv_dev[:20]

trainer = Trainer(
    model=model,
    args=training_args,  # reuse, doesn't matter much for eval-only
    data_collator=data_collator,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

metrics = trainer.evaluate()
print(metrics)'''